## Serialize Tabular Data using SchemaOrg and the CUAHSI.org-ScientificDataset extension

The purpose of this notebook is to evaluate how Raster data can be extracted and mapped to our Pydantic classes. It demonstrates how three common formats can be encoded in the CUAHSI.org ScientificDataset class: `CSV`, `Parquet`.


Each of these data formats is mapped to the ScientificDataset class via the following relationships:


|ScientificDataset Attribute|What it Describes|Example|
|---|---|---|
|Dimension	|The number of feature in the dataset	|rows = 12, columns=10, bands=5 |
|Variable	|The data variable represented by the raster grid | elevation(band, row, col) |


In [1]:
#!pip install s3fs pyarrow boto3 -q

In [2]:
import os
import sys
import s3fs
import boto3
import pandas
import hashlib
import rasterio
import mimetypes
from glob import glob
from pyproj import CRS
from pathlib import Path
from urllib.parse import urlparse
from pandas.api.types import is_numeric_dtype

# add the parent directory to the path. This is the 
# directory that contains our pydantic classes.
sys.path.append('..')
from src import base
from src import core
from src import dataset
from src import datavariable

In [3]:
def compute_sha256(file_path: Path) -> str:
    """Computes the SHA256 hash of a file.

    Args:
        file_path: The path to the file.

    Returns:
        The hexadecimal representation of the SHA256 hash.
    """
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()



def parse_s3_url(url: str):
    """
    Parses an S3 URL and returns (bucket, key)
    Supports both s3://bucket/key and https://bucket.s3.amazonaws.com/key
    """
    parsed = urlparse(url)

    if parsed.scheme == 's3':
        bucket = parsed.netloc
        key = parsed.path.lstrip('/')
    elif parsed.netloc.endswith('.s3.amazonaws.com'):
        bucket = parsed.netloc.split('.s3.amazonaws.com')[0]
        key = parsed.path.lstrip('/')
    else:
        raise ValueError(f"Unsupported S3 URL format: {url}")

    return bucket, key
    
def hash_s3_store(url: str) -> str:
    """
    Computes a SHA256 hash of the ETags (MD5s) of all S3 objects under a prefix.
    This mimics a hash of the object contents without downloading them.
    
    NOTE: If objects were uploaded with multipart, ETag is not a true MD5. Since this is 
          an example function, I'll leave the task of developing a better solution for
          the future.
    """

    bucket_name, prefix = parse_s3_url(url)
    
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    sha256 = hashlib.sha256()

    for page in pages:
        for obj in sorted(page.get('Contents', []), key=lambda o: o['Key']):
            etag = obj['ETag'].strip('"')  # ETag is a quoted string
            if '-' in etag:
                print(f"Warning: Object {obj['Key']} is multipart; ETag is not MD5")
                continue  # or handle multipart differently
            md5_bytes = bytes.fromhex(etag)
            sha256.update(md5_bytes)

    return sha256.hexdigest()

In [4]:
# Add raster MIME types if not already present
mimetypes.add_type("text/csv", ".csv")
mimetypes.add_type("application/vnd.apache.parquet", ".parquet")

In [5]:
def read_into_dataframe(filepath, delimiter=',', comment='#', skiprows=[], header=0, parse_all_dates=False):
    df = pandas.read_csv(filepath, sep=delimiter, comment=comment, skiprows=skiprows, header=header)

    # convert date columns into datetime64
    if parse_all_dates:
        for col in df.columns:
            if ('date' in col.lower()) or ('time' in col.lower()):
                try:
                    converted = pandas.to_datetime(df[col], errors='raise')
                
                    # Check if a reasonable number of non-NaT values exist after conversion
                    if converted.notna().sum() > 0:
                        df[col] = converted
                    
                except (ValueError, TypeError):
                    # Not a date column
                    pass
    return df

def read_nwis(filepath):
    df = read_into_dataframe(filepath,
                             delimiter='\t',
                             comment='#',
                             header=0)

    # remove the column widths row
    df = df.iloc[1:].reset_index(drop=True)
    
    # create a tz-aware datetime index
    df["datetime"] = pandas.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M")
    df.set_index("datetime", inplace=True)

    return df
    

def encode_pandas_dataframe(df, filepath):

    # Build dataset dimensions.
    dimensions = []
    coordinates = []
    idx_name = 'index' if df.index.name is None else df.index.name
    
    # build dimension from index
    dimensions.append(
        datavariable.Dimension(
            name =  idx_name,
            shape = len(df.index)
        )
    )
    # build coordinate from index
    coordinates.append(
        datavariable.DataVariable(
            name = idx_name,
            dataType = str(df.index.dtype),
            minValue = str(df.index.min()),
            maxValue = str(df.index.max()),
            dimensions=idx_name,
        )
    )           

    # define a list of variable names for which we will not compute statistics. This
    # is because min, max, nodata don't make much sense for some of these.
    skip_stats_for_variables = ['geometry', 'geom'] 
    
    variables = []
    for col_name in df.columns:
        minValue = None
        maxValue = None
        if is_numeric_dtype(df[col_name]):
            minValue = str(df[col_name].min())
            maxValue = str(df[col_name].max())
        variables.append(
            datavariable.DataVariable(
                name =col_name,
                dimensions=[dimensions[0].name],
                dataType = str(df[col_name].dtype),
                minValue = minValue,
                maxValue = maxValue,
                
            )
        )

    files = []
    
    if str(filepath).startswith('s3://'):
        
        # get the size of the file
        fs = s3fs.S3FileSystem()
        info = fs.info(filepath)
        size_bytes = info['size']
        
        contentSize = f"{size_bytes / (1024**2):.2f} MB"
        files = [
            base.MediaObject(contentUrl = filepath,
                             name = filepath,
                             sha256 = hash_s3_store(url),
                             contentSize = contentSize,
                             encodingFormat = mimetypes.guess_type(Path(filepath))[0],
                            )]
            
    else:
        # get all file names that match the patter of the input filepath
        search_path = f"{'.'.join(filepath.split('.')[:-1])}.*"
        associated_files = glob(search_path)
        
        for fpath in associated_files:
            files.append(
                base.MediaObject(
                    contentUrl = f'https://hydroshare.org/my-resource/{fpath}',
                    name = Path(fpath).name,
                    sha256 = compute_sha256(Path(fpath)),
                    contentSize = f'{os.path.getsize(Path(fpath))/1024} KB',
                    encodingFormat = mimetypes.guess_type(Path(fpath))[0],
                )
            )

    return dataset.ScientificDataset(
        variableMeasured = variables,
        dimensions = dimensions,
        coordinates = coordinates,
        associatedMedia=files,
        additionalType=dataset.AdditionalType.TABULAR,
    )

### Encode a CSV File with Unknown Dimensions

This demonstrates how a generic file can be represented in our schema.

In [6]:
df = read_into_dataframe('./data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv')
meta = encode_pandas_dataframe(df, './data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv')
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "additionalType": "Tabular",
    "associatedMedia": [
        {
            "type": "MediaObject",
            "contentUrl": "https://hydroshare.org/my-resource/data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv",
            "encodingFormat": "text/csv",
            "contentSize": "16169.5390625 KB",
            "name": "LR_GC_C_SourceID_1_QC_0_Year_2014.csv",
            "sha256": "f6db340e040de6a9c513af04e11f8bdad3f476dc1c1ee1c0064918974ad1b437"
        }
    ],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "LocalDateTime",
            "dimensions": [
                "index"
            ],
            "dataType": "object"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "UTCOffset",
            "dimensions": [
            

### Encode a CSV File with Dimensions

This demonstrates how a dimensions can be extracted from the csv file with additional logic, if available.

In [7]:
def read_custom_csv(filepath):

    # read the header information
    header = []
    with open(filepath, 'r') as f:
        for line in f.readlines():
            if line[0] == '#':
                txt = line[1:].strip()
                if len(txt) > 0:
                    header.append(txt)

    # read variable information
    variable_info = {}
    for line in header:
        if line[0:6] == 'Column':

            # the first element will be used as the key            
            key = line.split('|')[0].split(':')[1].strip()
            variable_info[key] = {}
            
            for col_dat in line.split('|')[1:]:
                variable_info[key][col_dat.split(':')[0].strip()] = ':'.join(col_dat.split(':')[1:]).strip()
                               
   
    
    df = read_into_dataframe(filepath,
                             delimiter=',',
                             comment='#',
                             header=0)


    # create a datetime index
    df["DateTimeUTC"] = pandas.to_datetime(df['DateTimeUTC'], format="%Y-%m-%d %H:%M:%S")
    df.set_index("DateTimeUTC", inplace=True)

    # Build dataset dimensions.
    dimensions = []
    coordinates = []
    idx_name = 'index' if df.index.name is None else df.index.name
    
    # build dimension from index
    dimensions.append(
        datavariable.Dimension(
            name =  idx_name,
            shape = len(df.index)
        )
    )
    # build coordinate from index
    coordinates.append(
        datavariable.DataVariable(
            name = idx_name,
            dataType = str(df.index.dtype),
            minValue = str(df.index.min()),
            maxValue = str(df.index.max()),
            dimensions=idx_name,
        )
    )           
    
    variables = []
    for col in df.columns:       
        if col in variable_info:
            minValue = None
            maxValue = None
            if is_numeric_dtype(df[col]):
                minValue = str(df[col].min())
                maxValue = str(df[col].max())
            variables.append(
                datavariable.DataVariable(
                    name =col,
                    dimensions=dimensions[0].name,
                    dataType = str(df[col].dtype),
                    minValue = minValue,
                    maxValue = maxValue,
                    unit = variable_info[col]['VariableUnitsName'],
                    description = variable_info[col]['MethodDescription'],
                    noDataValue = variable_info[col]['NoDataValue'],
                )
            )
        else:
            variables.append(
                datavariable.DataVariable(
                    name = col,
                    dimensions=[dimensions[0].name],
                    dataType = str(df[col].dtype),
                )
            )
    # get all file names that match the patter of the input filepath
    search_path = f"{'.'.join(filepath.split('.')[:-1])}.*"
    associated_files = glob(search_path)
    files = []
    for fpath in associated_files:
        files.append(
            base.MediaObject(
                contentUrl = f'https://hydroshare.org/my-resource/{fpath}',
                name = Path(fpath).name,
                sha256 = compute_sha256(Path(fpath)),
                contentSize = f'{os.path.getsize(Path(fpath))/1024} KB',
                encodingFormat = mimetypes.guess_type(Path(fpath))[0],
            )
        )
    
    return dataset.ScientificDataset(
        variableMeasured = variables,
        dimensions = dimensions,
        coordinates = coordinates,
        associatedMedia=files,
        additionalType=dataset.AdditionalType.TABULAR,
    )

    



In [8]:
meta = read_custom_csv('./data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv')
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "additionalType": "Tabular",
    "associatedMedia": [
        {
            "type": "MediaObject",
            "contentUrl": "https://hydroshare.org/my-resource/data/LR_GC_C_SourceID_1_QC_0_Year_2014.csv",
            "encodingFormat": "text/csv",
            "contentSize": "16169.5390625 KB",
            "name": "LR_GC_C_SourceID_1_QC_0_Year_2014.csv",
            "sha256": "f6db340e040de6a9c513af04e11f8bdad3f476dc1c1ee1c0064918974ad1b437"
        }
    ],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "LocalDateTime",
            "dimensions": [
                "DateTimeUTC"
            ],
            "dataType": "object"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "UTCOffset",
            "dimensions": [
      

### Encode a USGS NWIS Tab Separated File

In [9]:
df = read_nwis('./data/usgs_nwis_gills_creek.txt')
meta = encode_pandas_dataframe(df, './data/usgs_nwis_gills_creek.txt')
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "additionalType": "Tabular",
    "associatedMedia": [
        {
            "type": "MediaObject",
            "contentUrl": "https://hydroshare.org/my-resource/data/usgs_nwis_gills_creek.txt",
            "encodingFormat": "text/plain",
            "contentSize": "28.771484375 KB",
            "name": "usgs_nwis_gills_creek.txt",
            "sha256": "9e3a7bf4a3969932ae8a0fceac7c19b39105a115eaa91e7900d55629a193e1df"
        }
    ],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "agency_cd",
            "dimensions": [
                "datetime"
            ],
            "dataType": "object"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "site_no",
            "dimensions": [
                "datetime"
           

### Encode Parquet

In [10]:
url = 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/MA_Wetlands.parquet'
df = pandas.read_parquet(url, engine="pyarrow")

In [11]:
meta = encode_pandas_dataframe(df, 's3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/MA_Wetlands.parquet')
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://hydroshare.org/schema",
    "type": "ScientificDataset",
    "additionalType": "Tabular",
    "associatedMedia": [
        {
            "type": "MediaObject",
            "contentUrl": "s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/MA_Wetlands.parquet",
            "encodingFormat": "application/vnd.apache.parquet",
            "contentSize": "343.50 MB",
            "name": "s3://us-west-2.opendata.source.coop/giswqs/nwi/wetlands/MA_Wetlands.parquet",
            "sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
        }
    ],
    "variableMeasured": [
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "ATTRIBUTE",
            "dimensions": [
                "index"
            ],
            "dataType": "object"
        },
        {
            "context": "https://hydroshare.org/schema",
            "type": "DataVariable",
            "name": "WETLA